# 變電所資料前處理

# 1. 讀取資料

In [1]:
import pandas as pd
import os

DATA_DIR = "raw_data"

# 載入合併後的原始資料
df = pd.read_csv(os.path.join(DATA_DIR, "bpa_transformer_data_merged.csv"), low_memory=False)

# Check
df.head()

,Out Datetime,In Datetime,Name,Voltage High (kV),Voltage Low (kV),Duration (minutes),Outage Type,Dispatcher Cause,District,Outage #,year,Cause Dispatch,Responsible System Dispatch,Cause,Responsible System,O&M District,Transmission Owner NERC TADS,Out Datetime (PPT),In Datetime (PPT)
0,07/16/2012 07:37,02/21/2013 16:20,McNary: 345/230kV Transformer 8,345.0,230.0,317383,Plan,Maintenance,TRI,181084,2013,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,07/16/2012 07:37,02/21/2013 16:20,Ross: 345/230kV Transformer 4,345.0,230.0,317383,Plan,Maintenance,LGV,181085,2013,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,11/15/2012 09:18,02/21/2013 15:53,Ponderosa: 500/230kV Transformer 1,500.0,230.0,141515,Plan,Construction,RED,181970,2013,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,01/03/2013 10:12,01/03/2013 10:12,Maple Valley: 345/230kV Transformer 1,345.0,230.0,0,Auto,Unknown,COV,182778,2013,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,01/03/2013 10:12,01/03/2013 10:12,Rocky Reach: 345/230kV Transformer 1,345.0,230.0,0,Auto,Unknown,WEN,182779,2013,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


# 2. 資料檢視
* 特徵欄位項目
* Duplicated Data Check

## 特徵欄位項目

In [2]:
# 各年份欄位一覽
for year, group in df.groupby("year"):
    cols = [c for c in group.dropna(axis=1, how="all").columns]
    print(f"{year}: {cols}")

2013: ['Out Datetime', 'In Datetime', 'Name', 'Voltage High (kV)', 'Voltage Low (kV)', 'Duration (minutes)', 'Outage Type', 'Dispatcher Cause', 'District', 'Outage #', 'year']
2014: ['Out Datetime', 'In Datetime', 'Name', 'Voltage High (kV)', 'Voltage Low (kV)', 'Duration (minutes)', 'Outage Type', 'District', 'Outage #', 'year', 'Cause Dispatch', 'Responsible System Dispatch']
2015: ['Out Datetime', 'In Datetime', 'Name', 'Voltage High (kV)', 'Voltage Low (kV)', 'Duration (minutes)', 'Outage Type', 'District', 'Outage #', 'year', 'Cause Dispatch', 'Responsible System Dispatch']
2016: ['Out Datetime', 'In Datetime', 'Name', 'Voltage High (kV)', 'Voltage Low (kV)', 'Duration (minutes)', 'Outage Type', 'District', 'Outage #', 'year', 'Cause', 'Responsible System']
2017: ['Out Datetime', 'In Datetime', 'Name', 'Voltage High (kV)', 'Voltage Low (kV)', 'Duration (minutes)', 'Outage Type', 'District', 'Outage #', 'year', 'Cause', 'Responsible System']
2018: ['Out Datetime', 'In Datetime', 'N

## Duplicated Data Check

In [3]:
# 重複值檢查
f"重複值 : {df.duplicated().sum()}"

'重複值 : 0'

# 3. Feature Selection

* 目標

    將各年份欄位名稱不一致的欄位統一，捨棄不需要的欄位

* 處理項目

    **A. Out Datetime / In Datetime 統一（2022～2023）**

    2022～2023 欄位名稱改為 `Out Datetime (PPT)` / `In Datetime (PPT)`，合併回 `Out Datetime` / `In Datetime`

    **B. Cause 欄合併**

    依優先順序合併，統一欄名為 `Cause`

    - `Cause`：直接記錄（2016+，優先）
    - `Cause Dispatch`：調度員判斷（2014～2015，fallback）
    - `Dispatcher Cause`：調度員判斷（2013，fallback）

    > 注意：Transformer 資料**沒有** `Field Cause` / `Cause Field`，僅有調度員判斷，與電網版不同

    **C. 捨棄不需要欄位**

    最終保留：`Out Datetime`、`In Datetime`、`Name`、`Voltage Low (kV)`、`Voltage High (kV)`、`Duration (minutes)`、`Outage Type`、`Cause`、`year`

* 特徵欄位說明

    | 欄位名稱 | 說明 | 保留 |
    |---|---|---|
    | `Out Datetime` | 停電開始時間 | ✓ |
    | `In Datetime` | 復電時間 | ✓ |
    | `Name` | 變壓器名稱 | ✓ |
    | `Voltage High (kV)` | 高側電壓（kV） | ✓ |
    | `Voltage Low (kV)` | 低側電壓（kV） | ✓  |
    | `Duration (minutes)` | 停電持續時間（分鐘） | ✓ |
    | `Outage Type` | 停電類型（Auto / Plan 等） | ✓ |
    | `Dispatcher Cause` | 調度員判斷原因（2013） | ✗（合併至 Cause） |
    | `Cause Dispatch` | 調度員判斷原因（2014～2015） | ✗（合併至 Cause） |
    | `Cause` | 停電原因（2016+） | ✓ |
    | `Responsible System Dispatch` | 責任系統（2014～2015） | ✗ |
    | `Responsible System` | 責任系統（2016+） | ✗ |
    | `District` | 地區（2013～2017） | ✗ |
    | `O&M District` | 地區（2018+） | ✗ |
    | `Outage #` | 停電事件編號 | ✗ |
    | `Transmission Owner NERC TADS` | 輸電業者（NERC TADS 格式） | ✗ |
    | `Out Datetime (PPT)` | 停電開始時間，PPT 時區（2022～2023） | ✗（合併至 Out Datetime） |
    | `In Datetime (PPT)` | 復電時間，PPT 時區（2022～2023） | ✗（合併至 In Datetime） |
    | `year` | 資料年份 | ✓ |

In [4]:
# A. Out Datetime / In Datetime 統一（2022～2023 欄位名稱不同）
df["Out Datetime"] = df["Out Datetime"].fillna(df["Out Datetime (PPT)"])
df["In Datetime"]  = df["In Datetime"].fillna(df["In Datetime (PPT)"])

# B. Cause 欄合併：Cause 優先，依序 fallback
df["Cause"] = (df["Cause"]
               .fillna(df["Cause Dispatch"])
               .fillna(df["Dispatcher Cause"]))

# C. 保留分析所需欄位
keep_cols = ["Name", "Out Datetime", "In Datetime",
             "Voltage High (kV)", "Voltage Low (kV)",
             "Duration (minutes)", "Outage Type", "Cause", "year"]
df = df[keep_cols]

# Check
df

,Name,Out Datetime,In Datetime,Voltage High (kV),Voltage Low (kV),Duration (minutes),Outage Type,Cause,year
0,McNary: 345/230kV Transformer 8,07/16/2012 07:37,02/21/2013 16:20,345.0,230.0,317383,Plan,Maintenance,2013
1,Ross: 345/230kV Transformer 4,07/16/2012 07:37,02/21/2013 16:20,345.0,230.0,317383,Plan,Maintenance,2013
2,Ponderosa: 500/230kV Transformer 1,11/15/2012 09:18,02/21/2013 15:53,500.0,230.0,141515,Plan,Construction,2013
3,Maple Valley: 345/230kV Transformer 1,01/03/2013 10:12,01/03/2013 10:12,345.0,230.0,0,Auto,Unknown,2013
4,Rocky Reach: 345/230kV Transformer 1,01/03/2013 10:12,01/03/2013 10:12,345.0,230.0,0,Auto,Unknown,2013
...,...,...,...,...,...,...,...,...,...
3183,Alvey: 230/115kV Transformer 4,03/07/2022 07:22,03/07/2022 16:24,230.0,115.0,542,Plan,Maintenance,2022
3184,Alvey: 230/115kV Transformer 3,03/08/2022 07:52,03/08/2022 18:31,230.0,115.0,639,Plan,Maintenance,2022
3185,Grand Coulee: 287/230kV Transformer 1,07/17/2017 05:00,NaN,287.0,230.0,still out,Plan,Maintenance,2023
3186,Olympia: 287/230kV Transformer 3,10/28/2019 11:42,NaN,287.0,230.0,still out,Plan,Maintenance,2023


# 4. 檢視資料內容

查看 `Outage Type`、`Cause`內容，確認資料是否混入其他值、並統計發生異常的情況，以做因素分類(將因素歸納到 7 個大類中)

In [5]:
# Outage Type 值統計
print("Outage Type:")
print(df["Outage Type"].value_counts())

# Cause 值統計
print("\nCause:")
print(df["Cause"].value_counts())

Outage Type:
Outage Type
Plan    2659
Auto     529
Name: count, dtype: int64

Cause:
Cause
Maintenance                   1470
Voltage Control                568
Construction                   219
Unknown                        150
Emergency                      145
Forced (Configuration)         100
Urgent                          93
Lightning                       89
Forced                          62
Switching                       55
Terminal Equipment Failure      44
Testing                         19
Not Reported                    15
Improper Relaying               14
Proximity/Other                 13
Power System Condition          10
Oper Plan/RTCA Reqd Action      10
Wind                             8
Weather                          8
Equipment/Miscellaneous          8
Imp Install/Design/Applica       8
Contamination                    8
Foreign Request                  7
Human Element                    7
Fiber Optic Work                 6
Load Control                     6

## 自動停電因素統計

In [8]:
# A. 篩選自動停電（Auto）
df_auto = df[df["Outage Type"] == "Auto"].copy()

# 篩選後 Cause 分布
print(df_auto["Cause"].value_counts().to_string())

Cause
Unknown                       150
Forced (Configuration)        100
Lightning                      89
Terminal Equipment Failure     44
Not Reported                   15
Improper Relaying              14
Maintenance                    13
Construction                   11
Power System Condition         10
Wind                            8
Weather                         8
Equipment/Miscellaneous         8
Imp Install/Design/Applica      8
Contamination                   8
Human Element                   7
Smoke                           6
Fire                            6
Sympathetic                     4
Tree cut                        3
Tree blown                      3
Substation Operations           3
Foreign Trouble                 2
Line Material Failure           2
Bird droppings                  1
Dispatcher                      1
Another Line                    1
TT Noise                        1
Kite                            1
Bird or Animal                  1
Tree    

# 5. Data Cleaning

* 目標 : 

    從合併後的資料中，篩選出適合分析的停電事件

* 處理步驟

    **A. 篩選自動停電（Auto）**

    僅保留非計畫性的自動停電事件，排除計畫性維修停電（`Plan`）

    > 注意：篩選後 Cause 分布將大幅改變，Maintenance / Construction / Testing / Switching 等計畫性原因預期會大量消失，Cause 標準化對應表應以篩選後的結果為準

    **B. 排除 still out**

    `Duration (minutes)` 為 `still out` 表示該停電事件在資料截止日仍未復電，無法取得完整復電時間，予以排除

    **C. Duration 轉數值**

    將 `Duration (minutes)` 欄轉為數值型態（`float`），非數值內容轉為 `NaN`

    **D. 排除瞬間停電（duration ≤ 0）**

    持續時間為 0 或負值者視為瞬間停電，不納入分析

    **E. 排除 Duration 為 NaN**

    轉型後仍為 `NaN` 者直接排除該筆資料

    **F. Cause 標準化**

    將自動停電的 Cause 值對應至以下類別，其餘歸為 `Other`

    | 標準類別 | 對應原始值 |
    |---|---|
    | `Unknown` | `Unknown`, `Not Reported` |
    | `Lightning` | `Lightning` |
    | `Wind` | `Wind`, `Tree blown` |
    | `Weather` | `Weather`, `Contamination`, `Smoke`, `Fire` |
    | `Foreign Trouble` | `Foreign Trouble`, `Bird droppings`, `Bird or Animal`, `Kite`, `Tree cut`, `Tree` |
    | `Equipment Failure` | `Terminal Equipment Failure`, `Equipment/Miscellaneous`, `Line Material Failure`, `Imp Install/Design/Applica`, `Improper Relaying` |
    | `Other` | 其餘所有（`Forced (Configuration)`, `Power System Condition`, `Human Element`, `Sympathetic`, `Another Line`, `Substation Operations`, `Dispatcher`, `TT Noise`, `Maintenance`, `Construction` 等） |

In [9]:
# A. 篩選自動停電（Auto）
df = df[df["Outage Type"] == "Auto"].copy()

# B. 排除 still out
df = df[df["Duration (minutes)"] != "still out"]

# C. Duration 轉數值
df["Duration (minutes)"] = pd.to_numeric(df["Duration (minutes)"], errors="coerce")

# D. 排除瞬間停電（duration ≤ 0）
df = df[df["Duration (minutes)"] > 0]

# E. 排除 Duration 為 NaN
df = df[df["Duration (minutes)"].notna()]

# F. Cause 標準化
CAUSE_MAP = {
    "Unknown":          ["Unknown", "Not Reported"],
    "Lightning":        ["Lightning"],
    "Wind":             ["Wind", "Tree blown"],
    "Weather":          ["Weather", "Contamination", "Smoke", "Fire"],
    "Foreign Trouble":  ["Foreign Trouble", "Bird droppings", "Bird or Animal",
                         "Kite", "Tree cut", "Tree"],
    "Equipment Failure":["Terminal Equipment Failure", "Equipment/Miscellaneous",
                         "Line Material Failure", "Imp Install/Design/Applica",
                         "Improper Relaying"],
}

# item 反推 key 
cause_lookup = {v: k for k, vs in CAUSE_MAP.items() for v in vs}

df["Cause"] = df["Cause"].map(cause_lookup).fillna("Other")

# Check
df["Cause"].value_counts()

Cause
Other                102
Equipment Failure     72
Unknown               50
Lightning             26
Weather               17
Wind                   7
Foreign Trouble        6
Name: count, dtype: int64

# 6. Feature Creation 新增停電事件跨越年份

In [10]:
# 解析時間欄位
df["Out Datetime"] = pd.to_datetime(df["Out Datetime"], errors="coerce")
df["In Datetime"]  = pd.to_datetime(df["In Datetime"],  errors="coerce")

# 產生停電事件經過的年份 set
def get_outage_years(row):
    out = row["Out Datetime"]
    inp = row["In Datetime"]
    # 不處理停電事件時間記錄為空值者
    if pd.isna(out):
        return None
    # 復電時間為空值，僅記錄停電時間年份
    if pd.isna(inp):
        return {int(out.year)}
    return set(range(int(out.year), int(inp.year) + 1))

df["outage_year"] = df.apply(get_outage_years, axis=1)

# Check：找出跨年事件
cross_year = df[df["outage_year"].apply(lambda x: len(x) > 1 if not pd.isna(x) else False)]
print(f"跨年事件數：{len(cross_year)} 筆")
cross_year[["Out Datetime", "In Datetime", "outage_year"]].head(10)

跨年事件數：3 筆


,Out Datetime,In Datetime,outage_year
1502,2015-10-12 05:18:00,2017-10-26 15:18:00,"{2016, 2017, 2015}"
1902,2017-08-01 05:57:00,2018-02-23 13:14:00,"{2017, 2018}"
2828,2020-03-01 08:59:00,2021-01-26 16:59:00,"{2020, 2021}"


#  7. Check Missing Data

In [11]:
print("缺失值：")
print(df.isnull().sum())

缺失值：
Name                  0
Out Datetime          0
In Datetime           0
Voltage High (kV)     0
Voltage Low (kV)      0
Duration (minutes)    0
Outage Type           0
Cause                 0
year                  0
outage_year           0
dtype: int64


# 8. 異常值處理

* 目標
    
    移除不合理的極端停電持續時間，確保分析資料的品質

* 處理方式

    使用 DOE-417 歷史資料中，WECC 區域各年份的最長復電時間作為篩選**異常值**之**閥值**

* 設計原則

    * 異常值處理並非要處理掉極端搶修時間的案件，完全同意有極端修復時間的案例，但在做資料分析時，發現有部分值大於 3 個月，這是不可能出現在搶修下的情況，判斷為電力公司採用其他方式復電，而非實際搶修時間，故予以排除

* 超過閥值者直接**移除**

* 當事件跨越年度時，採用跨越年度的最大閥值

* 閥值來源

    `data/doe_wecc.csv` — DOE-417 WECC 各年度最大復電時間（`duration_minutes`）

In [12]:
# 讀取 DOE-417 資料
df_doe = pd.read_csv("data/doe_wecc.csv")
# 各年度最大值 df 
df_threshold = df_doe.groupby("year")["duration_minutes"].max().reset_index()

# 轉換為 dict
threshold_dict = df_threshold.set_index("year")["duration_minutes"].to_dict()
# 取出歷年來最大值
global_max = max(threshold_dict.values())

# 依據 年份回傳閥值 func
def get_threshold(outage_year):
    # 若發生年份為空值，以全域最大值計算
    if pd.isna(outage_year):
        return global_max
    # 取出年份對應的閥值
    values = [threshold_dict.get(y) for y in outage_year if threshold_dict.get(y) is not None]
    # 以 歷年來最大值 與 該跨越年份最大值 取大者為閥值 (這算是後來改的方案，因為原先的設計是閥值依據年度改變)
    return max(max(values), global_max) if values else global_max

df["threshold"] = df["outage_year"].apply(get_threshold)

before = len(df)
# 移除大於閥值的資料
df = df[df["Duration (minutes)"] <= df["threshold"]].copy()
after = len(df)
print(f"global_max = {global_max:.0f} 分鐘 = {global_max/1440:.1f} 天")
print(f"移除異常值後: {before} → {after}，移除 {before - after} 筆")

global_max = 49487 分鐘 = 34.4 天
移除異常值後: 280 → 275，移除 5 筆


# 9. 存檔

In [18]:
output_dir = "data"
os.makedirs(output_dir, exist_ok=True)
df.to_csv(os.path.join(output_dir, "transformer_clean.csv"), index=False)